In [2]:
### Data Ingestion


### Document datastructure

from langchain_core.documents import Document


In [3]:
doc=Document(
    page_content="This is the main text content I a using to create RAG",
    metadata={
        "source": "example.txt",
        "pages":1,
        "author": "Nandini Amaravadi",
        "data_created": "2026-01-01",
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Nandini Amaravadi', 'data_created': '2026-01-01'}, page_content='This is the main text content I a using to create RAG')

In [4]:
## create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [ ]:
simple_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """
}

for filepath, content in simple_texts.items():
    with open(filepath, 'w',encoding='utf-8') as f:
        f.write(content)

print("Sample text files created")

Sample text files created


In [5]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document=loader.load()
print(document)


C:\Users\NANDINI\AppData\Local\Temp\ipykernel_22896\1358805475.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

## load all the text files in the directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", # Pattern too match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True 
)

In [7]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Pol

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks=text_splitter.split_documents(pdf_documents)

print(f"Pages: {len(pdf_documents)}")
print(f"Chunks: {len(chunks)}")
print(f"First chunk: {chunks[0].page_content[:300]}")

Pages: 43
Chunks: 173
First chunk: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aid


In [ ]:
type(pdf_documents)

### embedding and VectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\NANDINI\GenAI-Essentials\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
         Initialize the embedding manager
         
         Args:
           model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
      """Load the SentenceTransformer model"""
      try:
        print(f"Loading embedding model: {self.model_name}")
        self.model=SentenceTransformer(self.model_name)
        print(f"Model loadded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
      except Exception as e:
        print(f"Error loading model {self.model_name}: {e}")
        raise

    def generate_embeddings(self, texts:List[str])->np.ndarray:
      """Generate embeddings for a list of texts

      Args:
         List of text string to embed

      Returns:
          numpy array of embeddings with shape (len(texts), embedding_dim)
      """
      if not self.model:
        raise ValueError("Model not loaded")

      print(f"Generating embeddings for {len(texts)} texts")
      embeddings=self.model.encode(texts, show_progress_bar=True)
      print(f"Generated Embeddings with shape: {embeddings.shape}")
      return embeddings

    def get_sentence_embedding_dimension(self)->int:
      """Get the embedding dimension of the model"""
      if not self.model:
        raise ValueError("Model not loaded")
      return self.get_sentence_embedding_dimension()

### initialize the embedding manager

embedding_manager=EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2747.42it/s]


Model loadded successfully. Embedding dimension: 384


C:\Users\NANDINI\AppData\Local\Temp\ipykernel_22896\2676652960.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loadded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [18]:
class VectorStore:
    """Manages document embeddings in chromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory:str="../data/vector_store"):
        """ Initalise the vector store 
         
         Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.clint=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        """ Initialze the chromaDB client and colelction"""
        try:
            # create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store is initialised. collection:{self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initialising the vector store; {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings:np.ndarray):
        """Add documents and embeddings to the vector store 
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #prepare document text 
            documents_text.append(doc.page_content)

            #prepare embeddings
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()
vector_store


Vector store is initialised. collection:pdf_documents
Existing documents in collection: 0


In [21]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\attention.pdf', 'file_path': '..\\data\\pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Pol

In [30]:
texts=[doc.page_content for doc in chunks]

#generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)
#embeddings

# store it in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 173 texts


Batches: 100%|██████████| 6/6 [00:06<00:00,  1.09s/it]


Generated Embeddings with shape: (173, 384)
Adding 173 documents to vector store...
Successfully added 173 documents to vector store
Total documents in collection: 173
